# 🎯 SQL: Common Table Expressions (CTEs)

**Advanced SQL Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** Code readability, intermediate result sets, recursive hierarchical queries
- **Tags:** `sql` | `ctes` | `readability` | `interview-prep`

## 📖 The Core Concept in Plain Language

A Common Table Expression (CTE) is a temporary named result set that you can reference within a `SELECT`, `INSERT`, `UPDATE`, or `DELETE` statement. It's essentially a way to name a "subquery" so that you can reuse it cleanly without nesting code.

### Why This Matters

- **Readability:** Flattens nested subqueries into a linear, top-to-bottom pipeline.
- **Reusability:** You can reference the same CTE multiple times in your main query (DRY principle).
- **Recursion:** CTEs are the only standard SQL way to walk hierarchies (trees, org charts, network routing).

### The Key Insight

```sql
WITH sales_summary AS (
    SELECT region, SUM(amount) AS total FROM sales GROUP BY region
)
SELECT * FROM sales_summary WHERE total > 1000;
```

## 🔍 CTE Basic Syntax and Execution

In [ ]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------

def sql_executer(str):
    con.execute(str)

def sql_executer_printer(str):
    print (con.execute(str).df().to_string(index=False))   

In [ ]:
sql = """
CREATE TABLE IF NOT EXISTS employees AS
SELECT * FROM (VALUES
    (1, 'Alice',   'Engineering', 120000),
    (2, 'Bob',     'Engineering', 120000),
    (3, 'Charlie', 'Engineering', 110000),
    (4, 'Diana',   'Engineering',  95000),
    (5, 'Eve',     'Marketing',   105000),
    (6, 'Frank',   'Marketing',   105000),
    (7, 'Grace',   'Marketing',    88000),
    (8, 'Henry',   'Finance',     115000),
    (9, 'Ivy',     'Finance',     115000),
    (10, 'Jack',   'Finance',      92000)
) t(id, name, department, salary)
"""
sql_executer(sql)
sql = """
CREATE TABLE IF NOT EXISTS server_metrics AS
SELECT * FROM (VALUES
    ('server-01', DATE '2026-01-01', 55.0),
    ('server-01', DATE '2026-01-02', 62.0),
    ('server-01', DATE '2026-01-03', 58.0),
    ('server-01', DATE '2026-01-04', 71.0),
    ('server-01', DATE '2026-01-05', 89.0),
    ('server-01', DATE '2026-01-06', 94.0),
    ('server-01', DATE '2026-01-07', 88.0),
    ('server-02', DATE '2026-01-01', 40.0),
    ('server-02', DATE '2026-01-02', 45.0),
    ('server-02', DATE '2026-01-03', 43.0),
    ('server-02', DATE '2026-01-04', 50.0),
    ('server-02', DATE '2026-01-05', 48.0),
    ('server-02', DATE '2026-01-06', 52.0),
    ('server-02', DATE '2026-01-07', 55.0)
) t(server_id, collection_date, cpu_utilization)
"""
sql_executer(sql)

sql="""
CREATE TABLE IF NOT Exists telemetry AS
SELECT * FROM (VALUES
    ('srv-01', '2026-03-01 08:00:00', 50.0, 'ACTIVE'),
    ('srv-01', '2026-03-01 16:00:00', 80.0, 'ACTIVE'),
    ('srv-01', '2026-03-02 08:00:00', 90.0, 'ACTIVE'),
    ('srv-01', '2026-03-02 16:00:00', 95.0, 'ACTIVE'),
    ('srv-02', '2026-03-01 10:00:00', 10.0, 'RETIRED'), -- Should be ignored!
    ('srv-03', '2026-03-01 08:00:00', 20.0, 'ACTIVE'),
    ('srv-03', '2026-03-01 16:00:00', 30.0, 'ACTIVE')
) t(server_id, collected_at, cpu_utilization, status)
"""

sql_executer(sql)

In [ ]:
# Demonstrating basic CTE syntax structure
# -------------------------------------------------------
# A CTE is defined using the WITH keyword before the main query.

basic_cte_sql = """
WITH top_employees AS (
    SELECT id, name, salary
    FROM employees
    WHERE salary > 100000
)
SELECT *
FROM top_employees
ORDER BY salary DESC;
"""

print("Syntax:")
print(basic_cte_sql)

## 🏗️ The Anatomy of Chained CTEs

You can define multiple CTEs in a single `WITH` clause by separating them with a comma. Later CTEs can reference earlier ones.

### Structure:
```sql
WITH step_1 AS (
    -- filter raw data
),
step_2 AS (
    -- aggregate step_1  <-- References step_1
)
SELECT * FROM step_2;  <-- Final query
```

In [ ]:
# Example of a chained pipeline
chained_cte_example = """
WITH raw_telemetry AS (
    SELECT server_id, DATE(collected_at) AS col_date, cpu_utilization
    FROM telemetry
    WHERE status = 'ACTIVE'
),
daily_averages AS (
    SELECT server_id, col_date, AVG(cpu_utilization) AS avg_cpu
    FROM raw_telemetry
    GROUP BY server_id, col_date
)
SELECT server_id, MAX(avg_cpu) AS peak_daily_avg
FROM daily_averages
GROUP BY server_id;
"""
print(chained_cte_example)

## 🔄 CTEs vs Subqueries

They often produce the same execution plan, but one is a nightmare to read.

In [ ]:
# -------------------------------------------------------
# Approach 1: Nested Subquery (Spaghetti Code)
# -------------------------------------------------------
subquery_sql = """
SELECT server_id, MAX(avg_cpu) 
FROM (
    SELECT server_id, DATE(collected_at) AS col_date, AVG(cpu_utilization) AS avg_cpu
    FROM (
        SELECT * FROM telemetry WHERE status = 'ACTIVE'
    ) active_telemetry
    GROUP BY server_id, DATE(collected_at)
) daily
GROUP BY server_id;
"""
print("\n✨ Approach 1 (Nightmare to parse):")
print(subquery_sql)

# -------------------------------------------------------
# Approach 2: CTEs (Linear logic)
# -------------------------------------------------------
print("\n✨ Approach 2 (Read top-to-bottom):")
print("WITH step1 AS (...), step2 AS (...) SELECT ...")

print(f"\n🔍 Both approaches identical: Yes, optimizer treats them exactly the same in most modern RDBMS.")

## 🌍 Real-World Example: Multi-step Aggregation

**The Citi Context:** At Citi, we had 6,000+ endpoints sending telemetry. Writing a single query to get the peak of the daily averages involves three steps: filter, daily aggregate, peak aggregate.

### Without CTEs (Nested Subqueries)
```sql
SELECT ... FROM ( SELECT ... FROM ( SELECT ... ) ) 
```

### With CTEs (Clean Pipeline)
```sql
WITH deduped AS (
    SELECT DISTINCT server_id, collection_date, cpu_utilization
    FROM telemetry_raw
    WHERE collection_date >= CURRENT_DATE - INTERVAL '30 days'
),
daily_avg AS (
    SELECT server_id, collection_date, AVG(cpu_utilization) AS avg_cpu
    FROM deduped
    GROUP BY server_id, collection_date
)
SELECT server_id, MAX(avg_cpu) AS peak_30day
FROM daily_avg
GROUP BY server_id
ORDER BY peak_30day DESC;
```

In [ ]:
# The impact of this pattern
print("- Readability: Top-to-bottom pipeline is instantly understandable.")
print("- Reusability: We can SELECT * FROM deduped while debugging to verify that step.")
print("- Modularity: Easy to copy/paste single CTE blocks into separate queries.")

## 🎭 Recursive CTEs

In [ ]:
# Recursive CTEs walk hierarchies (e.g., org charts)
# -------------------------------------------------------
recursive_cte = """
WITH RECURSIVE org_chart AS (
    -- Base case: find the CEO (no manager)
    SELECT id, name, manager_id, 1 AS depth
    FROM employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive step: join employees to the CTE itself
    SELECT e.id, e.name, e.manager_id, oc.depth + 1
    FROM employees e
    JOIN org_chart oc ON e.manager_id = oc.id
)
SELECT * FROM org_chart ORDER BY depth;
"""
print("Recursive Hierarchy traversal algorithm:")
print(recursive_cte)

## 🎤 5 Interview Q&A

### Q1: What is the difference between a CTE and a subquery?

**Answer:** CTEs are named and reusable within the query. They are evaluated top-to-bottom. Subqueries are inline and cannot be referenced more than once. CTEs significantly improve readability for multi-step data pipelines.

---

### Q2: Do CTEs improve query performance?

**Answer:** Generally, no. Most modern query optimizers treat CTEs exactly like subqueries and inline them into the execution plan. If you need materialization for performance (e.g., an index, or to avoid recalculating a massive join 10 times in the same script), you should use a Temporary Table.

---

### Q3: How do Recursive CTEs work?

**Answer:** They consist of a base case (`SELECT` to start the tree) and a recursive step (`UNION ALL` joining the table to the CTE itself). They iterate until the recursive query returns an empty set. They are used for graph traversal, like finding all direct and indirect reports in an org chart.

---

### Q4: When would you use a CTE vs a Temp Table in a data warehouse pipeline?

**Answer:** I use a CTE for single-query readability. I use a Temp Table when the intermediate dataset is huge and I need to reference it across *multiple different statements* in a stored procedure, or when I need to build an index on the intermediate data before joining it to a massive fact table.

---

### Q5: What is a major risk when using a Recursive CTE?

**Answer:** Infinite loops. If there is a cycle in the data (e.g., A manages B, B manages A), the recursion will run forever until memory runs out. You must always use a depth limit (e.g., `WHERE depth < 10`) or use the database's max recursion hints.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **CTE** | Common Table Expression; a temporary named result set | `WITH cte AS (...)` |
| **Recursive CTE** | A CTE that references itself to query hierarchical data | `WITH RECURSIVE...` |
| **Base Case** | The non-recursive part of a recursive query | `SELECT * FROM tbl WHERE parent IS NULL` |
| **`UNION ALL`** | The glue between the base case and recursive step | Combines results without removing duplicates |
| **Materialization** | Storing logic results physically to disk/RAM temporarily | Temp Tables materialize; CTEs usually don't. |
| **Temporary Table** | A table living only for the session duration | `CREATE TEMP TABLE x AS...` |

## 💼 The Citi Narrative: Simplifying Pipeline Logic

### The Problem

At Citi, dealing with telemetry from 6,000 agents meant queries often had 4 or 5 levels of aggregation (filter nulls -> daily avg -> 7-day rolling -> peak finding). We inherited scripts from senior analysts that were massively nested subqueries. They were impossible to review or debug.

### The Solution

I rewrote the core reporting pipelines using chained CTEs. It transformed "spaghetti SQL" into a modular, top-to-bottom data pipeline.

### The Code Pattern

```sql
WITH clean_data AS ( ... ),
daily_metrics AS ( SELECT ... FROM clean_data ),
rolling_stats AS ( SELECT ... FROM daily_metrics )
SELECT * FROM rolling_stats;
```

### The Impact

- **Maintainability:** New engineers could understand the entire pipeline in minutes instead of hours.
- **Debugging:** To test step 2, we just changed the final line to `SELECT * FROM daily_metrics WAIT LIMIT 10`. Instant visibility.
- **Code Reviews:** Errors were isolated to specific logical blocks rather than buried 3 levels deep.

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: Structuring a CTE
# Re-write this mental thought process into a CTE structure:
# 1. Find all users in 'NY'.
# 2. Calculate sum of their orders.
# 3. Select those > $500.

print("WITH ny_users AS ( SELECT user_id FROM users WHERE state='NY' ), \n" +
      "order_totals AS ( SELECT user_id, SUM(amount) as total FROM orders WHERE user_id IN (SELECT user_id FROM ny_users) GROUP BY user_id ) \n" +
      "SELECT * FROM order_totals WHERE total > 500;")

In [ ]:
# EXERCISE 2: Hierarchy
# How do you prevent infinite loops in a Recursive CTE?

print("Add a depth tracker in the SELECT, and add a WHERE depth < X in the recursive step.")

In [ ]:
# EXERCISE 3: Temp Table vs CTE
# You need to join the filtered telemetry log to 5 different dimension tables 
# in 5 separate insert statements. What do you use?

print("Temp table. A CTE only exists for a single statement. If you need it 5 times, a temp table avoids executing that filter logic 5 times.")

## 🎯 Summary: Key Takeaways

### What You've Learned

1. ✅ **CTE:** A named temporary result set using `WITH`.
2. ✅ **Readability First:** Transforms nested logic into linear pipelines.
3. ✅ **Performance Neutral:** RDBMS treat them like subqueries.
4. ✅ **Recursive:** The ONLY standard way to walk graph/tree data.
5. ✅ **Base Case:** The anchor of the recursion.
6. ✅ **UNION ALL:** Binds the base to the recursion.
7. ✅ **Cycles:** The danger of recursion; must be constrained.

### When to Use CTEs

- ✅ **DO** use for multi-step data pipelines within one query.
- ✅ **DO** use for recursive hierarchical logic.
- ✅ **DO** use to avoid writing the same subquery twice in one statement.
- ❌ **DON'T** use if you need the data across multiple separate `INSERT/UPDATE` statements (use temp tables).
- ❌ **DON'T** assume they act as an optimization fence or materialized cache.

### Interview Confidence Checklist

- [ ] Can write a chained CTE without looking up syntax.
- [ ] Can explain how CTEs differ from subqueries.
- [ ] Can explain how CTEs differ from Temp Tables.
- [ ] Understand the `WITH RECURSIVE` base case + UNION ALL structure.
- [ ] Can cite the Citi "from nested pasta to clean pipeline" story.
- [ ] Know how to prevent infinite loops in recursion.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

CTEs separate the junior scripters from the senior data engineers who write maintainable pipelines. 🚀